# 6.3 最终项目 - 智能知识库Agent

组装1.4-6.2的所有模块, 构建完整的RAG+Agent系统

## 项目架构
```
用户提问 -> Security -> Conversation -> HybridSearch -> ReActAgent -> 回答
                      |                  |                |
                  对话管理           BM25+Vector+RRF    工具调用+推理
```

In [ ]:
import sys, os
sys.path.insert(0, "..")
from config import get_client; client = get_client()
from agent_project import *
from agent_project.tools import create_default_registry
print(f"LLM: {client.name} | project modules loaded")


In [ ]:
from agent_project.app import KnowledgeBaseAgent
from agent_project.pipeline import DocumentPipeline
from agent_project.security import SecurityManager
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
emb = SentenceTransformer("BAAI/bge-small-zh-v1.5")

# Step 1: Load knowledge base
knowledge = [
    "Transformer由Vaswani等人在2017年提出,核心是Self-Attention机制。完全并行化处理序列,训练效率远超RNN。",
    "RAG(Retrieval-Augmented Generation)检索增强生成: 先检索相关文档,再让LLM基于文档生成答案。有效减少幻觉。",
    "DeepSeek V4于2026年4月发布,1.6T参数MoE架构(37B活跃),全栈昇腾910C训练。API成本约0.87元/百万输出tokens。",
    "Qwen3.7-Max于2026年5月发布,235B-A22B MoE架构,支持1M上下文,原生多模态(视觉+音频+工具),中文能力#1。",
    "MCP(Model Context Protocol)是AI Agent标准化协议,2025年12月捐赠Linux Foundation。定义Tool/Resource/Prompt三种原语。",
    "GLM-5.2于2026年6月发布,自研GLM架构,Agent能力T0级。在工具调用和复杂任务规划上表现优异。",
    "混合检索(Hybrid Search)将BM25关键词检索和向量语义检索结合,通过RRF融合排名,2026年RAG系统标配。",
    "LoRA(Low-Rank Adaptation)通过训练低秩矩阵实现高效微调。在Qwen3.7-1.7B上,仅需训练约1.6M参数(原参数的0.09%)。",
]
print(f"Knowledge base: {len(knowledge)} documents")


In [ ]:
# Step 2: Create the complete Agent
agent = KnowledgeBaseAgent(
    llm_client=client,
    embedding_model=emb,
    knowledge_docs=knowledge,
)
print("KnowledgeBaseAgent ready!")
print(f"  Tools: {agent.tools.list_tools()}")
print(f"  Security: injection+PII+rate_limit+audit")


In [ ]:
# Step 3: Test the Agent
print("=" * 60)
print("  TEST 1: Simple QA")
print("=" * 60)
result = agent.ask("What is MCP protocol?", verbose=True)
print(f"\nAnswer: {result['answer'][:300]}")
print(f"Sources: {len(result['sources'])} | Assessment: {result['assessment']['grade']}")


In [ ]:
print("\n" + "=" * 60)
print("  TEST 2: Multi-turn Conversation")
print("=" * 60)
result2 = agent.ask("What are its advantages?", verbose=True)
print(f"\nAnswer: {result2['answer'][:300]}")
print(f"Conversation turns: {agent.conversation.turn_count}")


In [ ]:
print("\n" + "=" * 60)
print("  TEST 3: Agent Mode (with tools)")
print("=" * 60)
result3 = agent.ask("Search for DeepSeek V4 info AND calculate 25*4+10", use_agent=True, verbose=True)
print(f"\nAnswer: {result3['answer'][:300]}")
print(f"Agent steps: {len(result3['trace'])}")


In [ ]:
# Step 4: System stats
stats = agent.get_stats()
print("\n" + "=" * 60)
print("  SYSTEM STATS")
print("=" * 60)
for k, v in stats.items():
    print(f"  {k}: {v}")
print("\n*** Intelligent Knowledge Base Agent - Project Complete! ***")
